In [1]:
import math
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(42)
torch.set_printoptions(precision=3, sci_mode=False)

In [2]:
B, N, Dv = 2, 6, 8
M, Dq, Dlm = 3, 12, 16

visual_tokens = torch.randn(B, N, Dv)
summary = visual_tokens.mean(dim=1)

print("Visual sequence:", tuple(visual_tokens.shape))
print("Pooled summary:", tuple(summary.shape))

reordered_summary = visual_tokens.flip(1).mean(dim=1)

print("Mean unchanged after reordering:",
    torch.allclose(summary, reordered_summary, atol=1e-6)
)

Visual sequence: (2, 6, 8)
Pooled summary: (2, 8)
Mean unchanged after reordering: True


$$\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In [3]:
Q = torch.tensor([
    [1., 0.], [0., 1.]
])

K = torch.tensor([
    [2., 0.], [0., 2.], [1., 1.]
])

V = torch.tensor([
    [10., 0.], [0., 10.], [5., 5.]
])

weights = (Q @ K.T / math.sqrt(K.shape[1])).softmax(dim=-1)
context = weights @ V

print("Attention weights:\n", weights)
print("Collected information:\n", context)
print("Weight sums:", weights.sum(dim=-1))

Attention weights:
 tensor([[0.576, 0.140, 0.284],
        [0.140, 0.576, 0.284]])
Collected information:
 tensor([[7.180, 2.820],
        [2.820, 7.180]])
Weight sums: tensor([1., 1.])


In [4]:
class MiniQueryBridge(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.queries = nn.Parameter(
            torch.randn(1, M, Dq) * 0.02
        )
        self.self_attn = nn.MultiheadAttention(
            Dq, 3, batch_first=True
        )
        self.cross_attn = nn.MultiheadAttention(
            Dq, 3, kdim=Dv, vdim=Dv, batch_first=True
        )
        self.norm1 = nn.LayerNorm(Dq)
        self.norm2 = nn.LayerNorm(Dq)
        self.norm3 = nn.LayerNorm(Dq)
        
        self.ffn = nn.Sequential(
            nn.Linear(Dq, 2 * Dq),
            nn.GELU(),
            nn.Linear(2 * Dq, Dq)
        )
    
    def forward(self, image_features):
        q = self.queries.expand(
            image_features.shape[0], -1, -1
        )
        update, _ = self.self_attn(
            q, q, q, need_weights=False
        )
        q = self.norm1(q + update)
        
        update, attention = self.cross_attn(
            q, image_features, image_features
        )
        q = self.norm2(q + update)
        q = self.norm3(q + self.ffn(q))

        return q, attention

In [5]:
bridge = MiniQueryBridge()
query_features, attention = bridge(visual_tokens)

print("Query features:", tuple(query_features.shape))
print("Attention map:", tuple(attention.shape))

with torch.no_grad():
    longer_features, _ = bridge(torch.randn(B, 10, Dv))

print("With 10 visual tokens:", tuple(longer_features.shape))

Query features: (2, 3, 12)
Attention map: (2, 3, 6)
With 10 visual tokens: (2, 3, 12)
